In [ ]:
import numpy as np
from numpy import pi as π
from tqdm.notebook import tqdm, trange
import matplotlib.pyplot as plt
import ufl
import firedrake
from firedrake import Constant, exp, sqrt, inner, grad, div, dx as dζ, ds, dS
import irksome
from irksome import Dt

In [ ]:
nz = 16
mesh = firedrake.UnitIntervalMesh(nz)

In [ ]:
family = "DG"  # <- revise later
degree = 0
density_element = firedrake.FiniteElement(family, "interval", degree)
Q = firedrake.FunctionSpace(mesh, density_element)

thickness_element = firedrake.FiniteElement("R", "interval", 0)
R = firedrake.FunctionSpace(mesh, thickness_element)

velocity_element = firedrake.FiniteElement("CG", "interval", degree + 1)
V = firedrake.VectorFunctionSpace(mesh, velocity_element)

See equation 6 in the CFM paper.
I'm using values just for phase 1 of densification to test things.
The coefficient $c_0$ that they use is the accumulation rate in meters of water equivalent, so we need the conversion factor of $\rho_s/\rho_w$ in the rate constant.

In [ ]:
a_s = Constant(0.3)       # m / yr
ρ_s = Constant(350.0)     # kg / m^3
ρ_i = Constant(917.0)     # kg / m^3
ρ_w = Constant(1000.0)    # kg / m^3
T = Constant(243.0)       # °K

def rate_constant(T, ρ_s):
    R = Constant(8.314)   # kJ / (mol °K)
    Q = Constant(10.16)   # kJ / mol
    c_0 = Constant(11.0)  # 1 / m
    return c_0 * exp(-Q / (R * T)) * ρ_s / ρ_w

In [ ]:
def get_test_function(q):
    z, = ufl.algorithms.extract_coefficients(q)
    Z = z.function_space()
    w = firedrake.TestFunction(Z)
    return firedrake.replace(q, {z: w})

In [ ]:
def thickness_equation(h, a_s, a_b, ω):
    g = get_test_function(h)
    mesh = ufl.domain.extract_unique_domain(h)
    ν = firedrake.FacetNormal(mesh)

    F_surf = (-a_s + h * inner(ω, ν)) * g * ds((2,))
    F_bed = (a_b + h * inner(ω, ν)) * g * ds((1,))
    return Dt(h) * g * dζ + F_surf + F_bed

In [ ]:
def velocity_equation(ω, ρ, h, a_b, C):
    v = get_test_function(ω)

    F_cells = h * (-inner(ω, v.dx(0)) + C(ρ) / ρ * v[0]) * dζ
    F_bed = a_b * v[0] * ds((1,))
    F_surf = h * inner(ω, v) * ds((2,))

    return F_cells + F_bed + F_surf

In [ ]:
def density_equation(ρ, h, ω, ρ_s, a_s):
    ϕ = get_test_function(ρ)
    mesh = ufl.domain.extract_unique_domain(ρ)
    ν = firedrake.FacetNormal(mesh)
    ω_ν = firedrake.max_value(0, inner(ω, ν))

    F_cells = (Dt(h * ρ) * ϕ - inner(h * ρ * ω, grad(ϕ))) * dζ
    F_facets = avg(h) * (ρ("+") * ω_ν("+") * ϕ("+") + ρ("-") * ω_ν("-") * ϕ("-")) * dS
    F_surf = -ρ_s * a_s * ϕ * ds((2,))
    F_bed = h * ρ * ω_ν * ϕ * ds((1,))

    return F_cells + F_facets + F_surf + F_bed

### Thickness equation

Testing this in isolation for sanity preservation.

In [ ]:
h = firedrake.Function(R)
h.assign(10.0)

t = firedrake.Constant(0.0)
a_0 = Constant(1.0)
δa = Constant(0.5)
a_s = a_0 + δa * firedrake.sin(2 * π * t)
a_b = a_0

F = thickness_equation(h, a_s, a_b, Constant((0.0,)))
timestep = 1.0 / 12
dt = firedrake.Constant(timestep)

method = irksome.BackwardEuler()
solver = irksome.TimeStepper(F, method, t, dt, h)

final_time = 5.0
num_steps = int(final_time / timestep)
hs = [float(h)]

for step in trange(num_steps):
    solver.advance()
    hs.append(float(h))
    t.assign(t + dt)

In [ ]:
hs = np.array(hs)
fig, ax = plt.subplots()
ax.plot(hs);

### Velocity equation

Testing in isolation for sanity preservation.
Use some assumed firn density and check that the velocity is correct, including the case where there is no conversion of firn to ice at the column base.

In [ ]:
h = Constant(1.0)

ζ, = firedrake.SpatialCoordinate(mesh)
ρ = firedrake.exp(-ζ)

def testing_compaction(ρ):
    return ρ**2

ω = firedrake.Function(V)
a_b = Constant(1.0)
F = velocity_equation(ω, ρ, h, a_b, testing_compaction)
firedrake.solve(F == 0, ω)

In [ ]:
S = firedrake.FunctionSpace(mesh, velocity_element)
ω_0 = firedrake.Function(S).interpolate(ω[0])

fig, ax = plt.subplots()
firedrake.plot(ω_0, axes=ax);

In [ ]:
ω_exact_expr = -a_b - (1 - firedrake.exp(-ζ))
firedrake.norm(ω_0 - ω_exact_expr) / firedrake.norm(ω_0)

### Density equation

Testing in isolation for sanity preservation.
Use a fixed vertical velocity and check that the final profile is correct.

### Coupled thickness + density

Let us pray

In [ ]:
z = firedrake.Function(Z)
z.sub(0).assign(0.1)
z.sub(1).assign(ρ_s)

h, ρ = firedrake.split(z)

In [ ]:
ζ, = firedrake.SpatialCoordinate(mesh)

ω_s = -a_s / h
r = ρ_s / ρ_i

c = rate_constant(T)
ω_expr = ω_s * (r + (1 - r) * exp(c * h * (1 - ζ) / r))

ω = firedrake.Function(V)
ω.interpolate(firedrake.as_vector((ω_expr,)));

In [ ]:
r, q = firedrake.TestFunctions(Z)

h, ρ = firedrake.split(z)

a_c = Constant(2 * a_s)
ρ_c = Constant(550.0)
a_b = a_c * (ρ - ρ_c) / (ρ_i - ρ_c)

ρ_b = Constant((ρ_c + sqrt(ρ_c**2 + 4 * (ρ_i - ρ_c) * ρ_s * a_s / a_c)) / 2)
z.sub(1).interpolate((1 - ζ) * ρ_b + ζ * ρ_s)

F_h = (
    (Dt(h) - a_s + a_b) * r * dζ +
    h * ω[0] * r * ds((1,)) - 
    h * ω[0] * r * ds((2,))
)

ν = firedrake.FacetNormal(mesh)
ω_ν = firedrake.max_value(0, inner(ω, ν))

F_ρ = (
    (Dt(h * ρ) * q - inner(h * ρ * ω, grad(q))) * dζ +
    firedrake.avg(h) * (ρ("+") * ω_ν("+") * q("+") + ρ("-") * ω_ν("-") * q("-")) * dS +
    ρ * a_b * q * ds((1,)) -
    ρ_s * a_s * q * ds((2,))
)

F = F_h + F_ρ

In [ ]:
method = irksome.BackwardEuler()
timestep = 1 / 3600
dt = Constant(timestep)
t = Constant(0.0)
solver = irksome.TimeStepper(F, method, t, dt, z)

In [ ]:
final_time = 5.0
num_steps = int(final_time / timestep)
zs = [z.copy(deepcopy=True)]
for step in trange(num_steps):
    solver.advance()
    t.assign(t + dt)
    zs.append(z.copy(deepcopy=True))

In [ ]:
fig, ax = plt.subplots()

ρ = z.subfunctions[1]
firedrake.plot(zs[2].subfunctions[1], axes=ax);

### Coupled thickness, velocity, and density

Let us pray but with like a lot more zeal this time